# HireSafe – Neural Network Model

This notebook trains a Neural Network (TensorFlow/Keras) on the HireSafe dataset.
It is part of the multi-model comparison study where **Random Forest** achieves the highest accuracy and is used in production.

In [ ]:
!pip install imbalanced-learn tensorflow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

## 1. Load Dataset

In [ ]:
df = pd.read_csv("hiresafe_ml_dataset.csv")
df.head()

## 2. Assign Hiring Status Label

In [ ]:
def assign_status(row):
    if row["workforce_percentage"] > 18:
        return "Slowdown"
    elif row["workforce_percentage"] > 8:
        return "Freeze"
    elif row["revenue_growth_percent"] > 2:
        return "Hiring"
    else:
        return "Freeze"

df["hiring_status"] = df.apply(assign_status, axis=1)

# Remove leakage feature
df = df.drop(columns=["workforce_percentage"])
df.head()

## 3. Feature Engineering

In [ ]:
df["layoff_intensity"] = df["layoff_count"] / (df["market_cap_billion_usd"] + 1)

df["financial_stress"] = (
    -df["revenue_growth_percent"] +
    abs(df["stock_price_change_7d"])
)

df["funding_strength"] = (
    df["total_funding_million_usd"] /
    (df["market_cap_billion_usd"] + 1)
)

df.head()

## 4. Encode Categoricals & Scale Features

In [ ]:
df = pd.get_dummies(df, columns=['industry', 'country'], drop_first=True)

# Encode target
le = LabelEncoder()
df['hiring_status'] = le.fit_transform(df['hiring_status'])

# Split features & target
X = df.drop('hiring_status', axis=1)
y = df['hiring_status']

# Scale features (required for Neural Network)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

## 5. Handle Class Imbalance with SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
print(y_train.value_counts())

## 6. Build Neural Network Architecture

In [ ]:
model = Sequential()

model.add(Dense(128, activation="relu", input_shape=(X_train.shape[1],)))
model.add(Dropout(0.3))

model.add(Dense(64, activation="relu", kernel_regularizer=l2(0.03)))
model.add(Dropout(0.3))

model.add(Dense(32, activation="relu", kernel_regularizer=l2(0.03)))

model.add(Dense(3, activation="softmax"))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 7. Train the Neural Network

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

## 8. Evaluate Neural Network

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

nn_accuracy = accuracy_score(y_test, y_pred)
print("Neural Network Accuracy:", nn_accuracy)
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 9. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Neural Network – Confusion Matrix")
plt.show()

## 10. Training Loss Curve

In [ ]:
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Neural Network – Training vs Validation Loss')
plt.legend()
plt.show()

## 11. Model Comparison Summary

> The Neural Network is trained here to explore deep-learning alternatives.
> However, after comparing all models, **Random Forest** achieves the highest accuracy on this dataset and is used in the HireSafe production system.

In [ ]:
# Reference accuracy from Random Forest (hiresafe_main.ipynb)
rf_accuracy = 0.92  # update with actual value from hiresafe_main

comparison = pd.DataFrame({
    "Model": ["Neural Network", "Random Forest (Production)"],
    "Accuracy": [round(nn_accuracy, 4), rf_accuracy]
})

print(comparison.to_string(index=False))
print("\n✅ Random Forest selected as the best model for production.")